# 图像卷积

## 卷积与互相关运算

### 互相关运算

In [3]:
import torch
from torch import nn
from d2l import torch as d2l

def corr2d(X, K):
    '''计算二维互相关运算'''
    h, w = K.shape
    Y = torch.zeros(X.shape[0]-h+1, X.shape[1]-w+1) # 输出形状
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i:i+h, j:j+w]*K).sum() # 计算互相关
    return Y

#### 验证上述二维互相关运算的输出

In [4]:
X = torch.tensor([[0, 1, 2], [3, 4, 5], [6, 7, 8]])
K = torch.tensor([[0, 1], [2, 3]])
Y = corr2d(X, K)
print(Y)

tensor([[19., 25.],
        [37., 43.]])


### 实现二维卷积层

In [5]:
class Conv2D(nn.Module):
    '''二维卷积层'''
    def __init__(self, kernel_size):
        super(Conv2D, self).__init__()
        self.weight = nn.Parameter(torch.rand(kernel_size)) # torch.rand(kernel_size)为一个长为kernel_size的向量，不是长宽都是kernel_size的矩阵
        self.bias = nn.Parameter(torch.rand(1))

    def forward(self, X):
        return corr2d(X, self.weight) + self.bias

#### 卷积层的一个简单应用

In [6]:
X = torch.ones(6, 8) # 输入形状为(6, 8)
X[:, 2:6] = 0 # 在X的第2到第5列设置为0
X

tensor([[1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.]])

In [7]:
K = torch.tensor([[1, -1]]) # 卷积核

#### 输出Y中的1代表从白色到黑色的边缘，-1代表黑色到白色的边缘

In [8]:
Y = corr2d(X, K) # 计算互相关
Y

tensor([[ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.]])

#### 卷积核K只能检测垂直边缘

In [9]:
print(corr2d(X.T, K))
print(corr2d(X, K.T))
print(corr2d(X.T, K.T)) # 输出形状

tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])
tensor([[0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.]])
tensor([[ 0.,  0.,  0.,  0.,  0.,  0.],
        [ 1.,  1.,  1.,  1.,  1.,  1.],
        [ 0.,  0.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  0.,  0.,  0.],
        [-1., -1., -1., -1., -1., -1.],
        [ 0.,  0.,  0.,  0.,  0.,  0.]])


### 学习由X生成Y的卷积核

In [10]:
conv2d = nn.Conv2d(1, 1, kernel_size=(1, 2), bias=False) # 创建一个卷积层
X = X.reshape((1, 1, 6, 8)) # 调整输入形状为(1, 1, 6, 8)
Y = Y.reshape((1, 1, 6, 7)) # 调整输出形状为(1, 1, 5, 7)

for i in range(10):
    Y_hat = conv2d(X)
    loss = (Y_hat-Y)**2
    conv2d.zero_grad() # 清除梯度
    loss.sum().backward() # 计算梯度
    conv2d.weight.data[:] -= 3e-2 * conv2d.weight.grad # 更新权重
    if (i+1) % 2 == 0:
        print(f'epoch {i+1}, loss {loss.sum():.3f}, weight {conv2d.weight.data.reshape(1,2)}')

epoch 2, loss 12.684, weight tensor([[ 0.2949, -0.4036]])
epoch 4, loss 2.195, weight tensor([[ 0.6987, -0.7682]])
epoch 6, loss 0.396, weight tensor([[ 0.8686, -0.9131]])
epoch 8, loss 0.078, weight tensor([[ 0.9410, -0.9695]])
epoch 10, loss 0.018, weight tensor([[ 0.9726, -0.9908]])


### 练习

1. 构建一个具有对角线边缘的图像`X`。
    1. 如果将本节中举例的卷积核`K`应用于`X`，会发生什么情况？
    1. 如果转置`X`会发生什么？
    1. 如果转置`K`会发生什么？
1. 在我们创建的`Conv2D`自动求导时，有什么错误消息？
1. 如何通过改变输入张量和卷积核张量，将互相关运算表示为矩阵乘法？
1. 手工设计一些卷积核。
    1. 二阶导数的核的形式是什么？
    1. 积分的核的形式是什么？
    1. 得到$d$次导数的最小核的大小是多少？


## 填充和步幅

### 在所有侧边填充1个像素

In [11]:
import torch
from torch import nn

def comp_conv2d(conv2d,X):
    X = X.reshape((1, 1) + X.shape)
    Y = conv2d(X)
    return Y.reshape(Y.shape[2:])

conv2d = nn.Conv2d(1, 1, kernel_size=3, padding=1)
X = torch.rand(size=(8, 8))
Y = comp_conv2d(conv2d, X)
print(Y.shape)  # torch.Size([8, 8])


torch.Size([8, 8])


### 填充不同的高度和宽度

In [12]:
conv2d = nn.Conv2d(1, 1, kernel_size=(5, 3), padding=(2, 1))  # (2, 1)是为了让输出和输入的高宽相同, 2为行, 1为列
comp_conv2d(conv2d, X).shape

torch.Size([8, 8])

### 将高度和宽度的步幅设置为2

In [13]:
conv2d = nn.Conv2d(1, 1, kernel_size=3, padding=1, stride=2)
Y = comp_conv2d(conv2d, X)
print(Y.shape)  # torch.Size([4, 4])

torch.Size([4, 4])


#### 一个稍微复杂的例子：

In [14]:
conv2d = nn.Conv2d(1, 1, kernel_size=(3, 5), padding=(0, 1), stride=(3, 4))  # (0, 1)是为了让输出和输入的高宽相同, 0为行, 1为列
comp_conv2d(conv2d, X).shape

torch.Size([2, 2])

## 多输入多输出通道

### 实现一下多输入通道互相关运算

In [28]:
import torch
from torch import nn
from d2l import torch as d2l

def corr2d_multi_in(X, K):
    return sum(d2l.corr2d(x, k) for x, k in zip(X, K))

# 验证互相关运算的输出
X = torch.tensor([[[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]],
                  [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]]])
K = torch.tensor([[[0.0, 1.0], [2.0, 3.0]], [[1.0, 2.0], [3.0, 4.0]]])
print(corr2d_multi_in(X, K))

tensor([[ 56.,  72.],
        [104., 120.]])


### 计算多个通道的输出的互相关运算

In [29]:
def corr2d_multi_in_out(X, K):
    return torch.stack([corr2d_multi_in(X, k) for k in K], 0)
# 验证多输入通道和多输出通道的互相关运算
K = torch.stack((K+1, K+2, K+3), 0)
print(K.shape)

corr2d_multi_in_out(X, K)

torch.Size([3, 2, 2, 2])


tensor([[[ 76., 100.],
         [148., 172.]],

        [[ 96., 128.],
         [192., 224.]],

        [[116., 156.],
         [236., 276.]]])

### 1*1卷积

In [30]:
def corr2d_multi_in_out_1x1(X, K):
    c_i, h, w = X.shape
    c_o = K.shape[0]
    X = X.reshape(c_i, h*w)
    K = K.reshape(c_o, c_i)
    Y = torch.matmul(K, X)
    return Y.reshape(c_o, h, w)
# 验证1x1卷积层的互相关运算
X = torch.normal(0, 1, (3, 3, 3))
K = torch.normal(0, 1, (2, 3, 1, 1))

Y1 = corr2d_multi_in_out_1x1(X, K)
Y2 = corr2d_multi_in_out(X, K)
assert float(torch.abs(Y1-Y2).sum())<1e-6